- Load heart disease dataset in pandas dataframe
- Remove outliers using Z score. Usual guideline is to remove anything that has Z score > 3 formula or Z score < -3
- Convert text columns to numbers using label encoding and one hot encoding
- Apply scaling
- Build a classification model using support vector machine. Use standalone model as well as Bagging model and check if you see any difference in the performance.
- Now use decision tree classifier. Use standalone model as well as Bagging and check if you notice any difference in performance
- Comparing performance of svm and decision tree classifier figure out where it makes most sense to use bagging and why. Use internet to figure out in what conditions bagging works the best.


In [81]:
import pandas as pd
df = pd.read_csv('heart.csv')
df.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


In [82]:
df.isna().sum()

Age               0
Sex               0
ChestPainType     0
RestingBP         0
Cholesterol       0
FastingBS         0
RestingECG        0
MaxHR             0
ExerciseAngina    0
Oldpeak           0
ST_Slope          0
HeartDisease      0
dtype: int64

In [83]:
df.describe()

,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak,HeartDisease
count,918.000000,918.000000,918.000000,918.000000,918.000000,918.000000,918.000000
mean,53.510893,132.396514,198.799564,0.233115,136.809368,0.887364,0.553377
std,9.432617,18.514154,109.384145,0.423046,25.460334,1.066570,0.497414
min,28.000000,0.000000,0.000000,0.000000,60.000000,-2.600000,0.000000
25%,47.000000,120.000000,173.250000,0.000000,120.000000,0.000000,0.000000
50%,54.000000,130.000000,223.000000,0.000000,138.000000,0.600000,1.000000
75%,60.000000,140.000000,267.000000,0.000000,156.000000,1.500000,1.000000
max,77.000000,200.000000,603.000000,1.000000,202.000000,6.200000,1.000000


In [84]:
df.shape

(918, 12)

removing outliers

In [85]:
df = df[df.Cholesterol<(df.Cholesterol.mean()+3*df.Cholesterol.std())]
df.shape

(915, 12)

In [86]:
df = df[df.MaxHR<(df.MaxHR.mean()+3*df.MaxHR.std())]
df.shape

(915, 12)

In [87]:
df = df[df.Oldpeak<(df.Oldpeak.mean()+3*df.Oldpeak.std())]
df.shape

(909, 12)

In [88]:
df = df[df.RestingBP<(df.RestingBP.mean()+3*df.RestingBP.std())]
df.shape

(902, 12)

In [89]:
df.ChestPainType.unique()

<StringArray>
['ATA', 'NAP', 'ASY', 'TA']
Length: 4, dtype: str

In [90]:
df.RestingECG.unique()

<StringArray>
['Normal', 'ST', 'LVH']
Length: 3, dtype: str

In [91]:
df.ExerciseAngina.unique()

<StringArray>
['N', 'Y']
Length: 2, dtype: str

In [92]:
df.ST_Slope.unique()

<StringArray>
['Up', 'Flat', 'Down']
Length: 3, dtype: str

In [93]:
new_df = df.copy()
new_df.shape

(902, 12)

In [94]:
new_df.ST_Slope.replace({
    'Up':1,
    'Flat':2,
    'Down':3,
})
new_df.ExerciseAngina.replace({
    'N':0,
    'Y':1
})
new_df.RestingECG.replace({
    'Normal':1,
    'ST':2,
    'LVH':3,
})
new_df.shape

(902, 12)

In [95]:
df_f= pd.get_dummies(new_df, drop_first=True)
df_f.shape

(902, 16)

In [96]:
X = df_f.drop('HeartDisease',axis=1)
y = df_f.HeartDisease

In [97]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled

array([[-1.42896269,  0.46089071,  0.85238015, ..., -0.82065181,
        -1.00221976,  1.13805334],
       [-0.47545956,  1.5925728 , -0.16132855, ..., -0.82065181,
         0.99778516, -0.87869344],
       [-1.74679706, -0.10495034,  0.79657967, ..., -0.82065181,
        -1.00221976,  1.13805334],
       ...,
       [ 0.37209878, -0.10495034, -0.61703246, ...,  1.21854359,
         0.99778516, -0.87869344],
       [ 0.37209878, -0.10495034,  0.35947592, ..., -0.82065181,
         0.99778516, -0.87869344],
       [-1.64085227,  0.3477225 , -0.20782894, ..., -0.82065181,
        -1.00221976,  1.13805334]], shape=(902, 15))

In [98]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test  = train_test_split(X_scaled,y,test_size=0.2,random_state=0)

In [99]:
X_train.shape

(721, 15)

In [100]:
X_test.shape

(181, 15)

In [101]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
scores = cross_val_score(RandomForestClassifier(),X,y,cv=5)
scores.mean()

np.float64(0.8180908532842235)

In [102]:
from sklearn.tree import DecisionTreeClassifier

scores = cross_val_score(DecisionTreeClassifier(random_state=0), X, y, cv=5)
scores.mean()

np.float64(0.7293615715162678)

In [103]:
from sklearn.ensemble import BaggingClassifier
bag_model = BaggingClassifier(
    n_estimators=100,
    max_samples=100,
    random_state=0,
    oob_score=True,
)
scores = cross_val_score(BaggingClassifier(),X,y,cv=5)
scores.mean()

np.float64(0.7870779619398404)

In [80]:
bag_model = BaggingClassifier(
    estimator=RandomForestClassifier(random_state=0),
        n_estimators=100
)
scores = cross_val_score(bag_model,X,y,cv=5)
scores.mean()

np.float64(0.8269981583793738)